# Generic chemostat: forward simulation

Run all six steps to simulate the configured process and reproduce `outputs/trajectories.csv`, `trajectories.png`, and `trajectories.svg`. CSV files open in spreadsheet applications. Only `inputs/case.json`, `mechanism.json`, and `thermo.json` are required.

The current inputs select the additive working-volume policy with native continuous outlet control. Change inputs to choose another supported policy; plots are model predictions. All notebook steps are forward simulation and output generation.


## Step 1: Select input and output folders


In [ ]:
# STEP 1: Select input and output folders
# Run each cell in order with Shift+Enter, or use Run All.
# Change example_dir for a copied case. Exports replace matching files in EXPORT_DIR;
# choose another folder to preserve a previous run, or set WRITE_ARTIFACTS=False.
from pathlib import Path
import csv
import json
import sys
import numpy as np
import matplotlib.pyplot as plt

repo_root = next((path for path in (Path.cwd(), *Path.cwd().parents)
                  if (path / "PharmaPy").is_dir()), None)
if repo_root is None:
    raise ValueError("Set repo_root to your PharmaPy checkout before running this cell")
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))
from PharmaPy.Bioreactors import build_bioreactor
from PharmaPy.Bioreactors.input_format import normalize_forward_inputs

example_dir = repo_root / "examples/bioreactors/generic_chemostat"
input_dir = example_dir / "inputs"
thermo_path = input_dir / "thermo.json"
EXPORT_DIR = example_dir / "outputs"
WRITE_ARTIFACTS = True


## Step 2: Load and edit forward-simulation inputs


In [ ]:
# STEP 2: Load and edit forward-simulation inputs
# Edit case.json, mechanism.json and thermo.json, or edit these dictionaries below.
# Keep species identifiers and initial mass/volume/density consistent. Input units
# are declared in the JSON files; recipe times use numerics.step.unit.
# Rerun this and all following cells after changing inputs.
case = json.loads((input_dir / "case.json").read_text())
config = json.loads((input_dir / "mechanism.json").read_text())
# case["operation"]["volume_policy"] = "working-volume"
# case["operation"]["runtime"]["value"] = 6.0
print(json.dumps(case, indent=2))


## Step 3: Run the native forward simulation


In [ ]:
# STEP 3: Run the native forward simulation
# Run to construct a fresh reactor and execute its complete configured schedule.
# histories contains every segment, including both sides of material additions.
assembly = build_bioreactor(case, config, thermo_path)
unit = assembly.unit
histories = assembly.solve(verbose=False)
print("Native reactor:", type(unit).__name__)
print("Recorded segments:", len(histories))


## Step 4: Build the trajectory table


In [ ]:
# STEP 4: Build the trajectory table
# Run to convert native inventories using each recorded reactor volume.
# Time and amount units appear in column names; CHO species columns are mmol/L.
# Duplicate event times retain pre/post-addition rows in segment order.
result = histories[0]
nutrient_index = list(assembly.phase.name_species).index("nutrient")
molecular_weight = float(assembly.phase.mw[nutrient_index])
time_h = np.asarray(result.time, dtype=float) / 3600.0
mass_kg = np.asarray(result.mass_j_liquid0, dtype=float)
biomass_kg = np.asarray(result.biomass_kg_liquid0, dtype=float).reshape(-1)
nutrient_mol = mass_kg[:, nutrient_index] * 1000.0 / molecular_weight
liquid_mass_kg = mass_kg.sum(axis=1)
volume_l = np.asarray(result.vessel_vol).reshape(-1) * 1000.0
rows = [dict(time_h=float(t), nutrient_mmol=float(n * 1000.0),
             biomass_gdw=float(b * 1000.0), liquid_volume_l=float(volume),
             nutrient_mmol_l=float(n * 1000.0 / volume),
             biomass_gdw_l=float(b * 1000.0 / volume),
             liquid_plus_biomass_kg=float(mass + b))
        for t, n, b, volume, mass in zip(time_h, nutrient_mol, biomass_kg,
                                        volume_l, liquid_mass_kg)]
print("Initial row:", rows[0])
print("Final row:", rows[-1])


## Step 5: Plot the simulated trajectories


In [ ]:
# STEP 5: Plot the simulated trajectories
# Run to display the figure. Edit the species selection, axes and labels here.
# Curves use the same rows and recorded volumes as the exported trajectory table.
figure, axes = plt.subplots(2, 1, figsize=(8, 7), sharex=True, constrained_layout=True)
axes[0].plot(time_h, nutrient_mol * 1000.0, label="nutrient")
axes[0].set(ylabel="Nutrient (mmol)", title="Generic chemostat forward simulation")
axes[1].plot(time_h, biomass_kg * 1000.0 / volume_l, label="biomass")
axes[1].set(ylabel="Biomass (gDW/L)")
axes[-1].set(xlabel="Time (h)")
for axis in axes:
    axis.grid(alpha=0.25)
    axis.legend()
plt.show()


## Step 6: Export the trajectory spreadsheet and figures


In [ ]:
# STEP 6: Export the trajectory spreadsheet and figures
# Run to save a spreadsheet-compatible CSV plus PNG and SVG figures.
# Files in EXPORT_DIR with these names are replaced; no fitting or comparison runs.
if WRITE_ARTIFACTS:
    EXPORT_DIR.mkdir(parents=True, exist_ok=True)
    with (EXPORT_DIR / "trajectories.csv").open("w", newline="") as stream:
        writer = csv.DictWriter(stream, fieldnames=list(rows[0]), lineterminator="\n")
        writer.writeheader()
        writer.writerows(rows)
    figure.savefig(EXPORT_DIR / "trajectories.png", dpi=180)
    figure.savefig(EXPORT_DIR / "trajectories.svg")
    print("Saved trajectories.csv, trajectories.png and trajectories.svg to", EXPORT_DIR)
else:
    print("Export disabled; trajectories and figure remain in memory.")
